# 📧 Detecting Spam Emails
## Notebook 2: Text Preprocessing

**KKR Gen AI Innovations** | [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)

---

### 🎯 What You Will Learn
1. Why text needs to be cleaned before ML
2. How to remove noise (URLs, special characters, numbers)
3. What stopwords are and why we remove them
4. How stemming works
5. How to convert text to numbers (TF-IDF)

---

### 📌 Key Concept: Why Preprocess Text?

Raw text is messy. Consider these two messages:
- `"CLICK HERE NOW!!!! Win $$$"`
- `"click here now win"`

They mean the same thing, but a computer sees them as completely different.
Preprocessing standardizes text so our model can learn patterns more effectively.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Load our custom preprocessing module
from src.data_preprocessing import (
    load_dataset, clean_text, remove_stopwords,
    apply_stemming, preprocess_text, preprocess_dataframe
)
from src.utils import download_dataset

print('✅ Modules imported!')

## Step 1: Load the Dataset

In [ ]:
download_dataset('../data')
df = load_dataset('../data/spam.csv')
print(df.head())

## Step 2: Text Cleaning — Step by Step

Let's walk through each preprocessing step individually so you understand what happens.

In [ ]:
# Let's take a spam example
raw_text = "WINNER!! You have been selected to receive a £900 prize reward! Call NOW 09061701461"

print('📝 PREPROCESSING WALKTHROUGH')
print('='*60)
print(f'Original:     {raw_text}')
print()

# Step 1: Clean (lowercase, remove URLs, numbers, punctuation)
step1 = clean_text(raw_text)
print(f'After Clean:  {step1}')

# Step 2: Remove stopwords
step2 = remove_stopwords(step1)
print(f'No Stopwords: {step2}')

# Step 3: Stemming
step3 = apply_stemming(step2)
print(f'After Stem:   {step3}')

print('='*60)
print(f'Final Result: {step3}')
print(f'Original words: {len(raw_text.split())} → Final words: {len(step3.split())}')

## Step 3: Understanding Stopwords

**Stopwords** are common English words that don't help classify spam.
Words like 'the', 'is', 'at', 'which', 'on' appear in both spam AND ham equally.

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

stop_words = list(stopwords.words('english'))
print(f'Total English stopwords: {len(stop_words)}')
print(f'\nFirst 30 stopwords:')
print(stop_words[:30])

## Step 4: Understanding Stemming

**Stemming** reduces words to their root form so similar words are treated as the same.

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

# Show stemming examples
word_groups = [
    ['winning', 'winner', 'wins', 'won'],
    ['claiming', 'claims', 'claimed', 'claim'],
    ['clicking', 'clicked', 'clicks', 'click'],
    ['running', 'runner', 'runs', 'ran'],
]

print('📝 Stemming Examples:')
print('-'*40)
for group in word_groups:
    stemmed = [stemmer.stem(w) for w in group]
    print(f'  {group} → {stemmed}')

print('\nAll variations map to the SAME root!')

## Step 5: Apply Preprocessing to the Full Dataset

In [ ]:
# Process all messages
df = preprocess_dataframe(df)

print('\n📋 Preprocessed Data Sample:')
df[['label', 'message', 'cleaned_message', 'label_encoded']].head()

## Step 6: Convert Text to Numbers — TF-IDF

Machine learning models work with numbers, not text.
**TF-IDF** (Term Frequency-Inverse Document Frequency) converts text to a numerical matrix.

- **TF** (Term Frequency): How often a word appears in ONE message
- **IDF** (Inverse Document Frequency): How rare the word is ACROSS ALL messages
- **TF-IDF = TF × IDF**: Rare words get higher scores — they're more informative!

In [ ]:
from src.feature_extraction import split_data, extract_tfidf_features

# Split data: 80% train, 20% test
X = df['cleaned_message']
y = df['label_encoded']
X_train, X_test, y_train, y_test = split_data(X, y)

# Extract TF-IDF features
X_train_tfidf, X_test_tfidf, vectorizer = extract_tfidf_features(
    X_train.tolist(), X_test.tolist()
)

print(f'\n📊 Feature Matrix Info:')
print(f'   Training matrix shape: {X_train_tfidf.shape}')
print(f'   Each message is represented as a vector of {X_train_tfidf.shape[1]} numbers!')

In [ ]:
# Show top TF-IDF features for a spam message
import numpy as np

# Find first spam message in test set
spam_idx = np.where(y_test == 1)[0][0]
spam_message = X_test.iloc[spam_idx]
spam_features = X_test_tfidf[spam_idx]

# Get top features
feature_names = vectorizer.get_feature_names_out()
top_indices = np.argsort(spam_features.toarray()[0])[-10:][::-1]

print(f'📧 Spam message: "{spam_message[:80]}..."')
print(f'\nTop 10 TF-IDF features:')
for idx in top_indices:
    score = spam_features.toarray()[0][idx]
    if score > 0:
        print(f'  Word: "{feature_names[idx]}" | TF-IDF Score: {score:.4f}')

## ✅ Summary

After preprocessing, our data is ready for machine learning:

| Stage | Format | Example |
|-------|--------|----------|
| Raw text | String | `"FREE! WIN NOW!!"` |
| Cleaned | String | `"free win now"` |
| Stemmed | String | `"free win now"` |
| TF-IDF | Numbers | `[0, 0.5, 0.8, 0, 0.3, ...]` |

---

### ➡️ Next Step
Open **Notebook 3: Naive Bayes Model** to train our first classifier!

---
*KKR Gen AI Innovations | [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)*